In [1]:
# 04. Customer Segmentation Analysis

## 目的

顧客の購買行動を分析し、顧客ごとの購買規模・購買頻度・リピート状況を把握する。

本分析では、customers.csv と orders.csv を結合し、以下のKPIを算出する。

- 顧客数
- 総購入回数
- 総購入金額
- 顧客あたり購入金額
- 購入頻度
- リピート率

さらに、購買金額と購入頻度をもとに顧客をセグメント化し、
各セグメントの特徴と売上への貢献度を分析する。

※本分析に使用するデータは、公開情報や一般的な書店・EC市場の構造を参考に、
データ分析手法の実証を目的として設計した疑似データである。
実在企業の顧客情報・購買履歴・内部データではない。

SyntaxError: invalid character '、' (U+3001) (<ipython-input-1-93ef694b821d>, line 5)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
customers = pd.read_csv("dataset/customers.csv")
orders = pd.read_csv("dataset/orders.csv")

print("customers:", customers.shape)
print("orders:", orders.shape)

display(customers.head())
display(orders.head())

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/customers.csv'

In [4]:
print("=== customers ===")
customers.info()

print("\n=== orders ===")
orders.info()

=== customers ===


NameError: name 'customers' is not defined

In [5]:
customer_ids = set(customers["customer_id"])
order_customer_ids = set(orders["customer_id"])

missing_customers = order_customer_ids - customer_ids

print("ordersに存在するがcustomersに存在しないcustomer_id:")
print(missing_customers)

assert len(missing_customers) == 0, \
    "ordersにcustomers側で確認できないcustomer_idがあります。"

print("✓ 顧客IDの整合性チェック完了")

NameError: name 'customers' is not defined

In [6]:
customer_orders = (
    orders.groupby("customer_id")
    .agg(
        order_count=("order_id", "nunique"),
        total_quantity=("quantity", "sum"),
        total_spend=("unit_price", lambda x: (x * orders.loc[x.index, "quantity"]).sum()),
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max")
    )
    .reset_index()
)

display(customer_orders.head())

NameError: name 'orders' is not defined

In [7]:
customer_analysis = customers.merge(
    customer_orders,
    on="customer_id",
    how="left"
)

customer_analysis["order_count"] = (
    customer_analysis["order_count"].fillna(0)
)

customer_analysis["total_quantity"] = (
    customer_analysis["total_quantity"].fillna(0)
)

customer_analysis["total_spend"] = (
    customer_analysis["total_spend"].fillna(0)
)

display(customer_analysis.head())

NameError: name 'customers' is not defined

In [8]:
customer_analysis["customer_type"] = np.where(
    customer_analysis["order_count"] >= 2,
    "Repeat",
    "One-time"
)

customer_analysis["is_repeat"] = (
    customer_analysis["order_count"] >= 2
).astype(int)

display(
    customer_analysis[
        [
            "customer_id",
            "order_count",
            "total_spend",
            "customer_type"
        ]
    ].head(10)
)

NameError: name 'customer_analysis' is not defined

In [9]:
total_customers = len(customer_analysis)

repeat_customers = (
    customer_analysis["is_repeat"].sum()
)

repeat_rate = (
    repeat_customers / total_customers
)

total_spend = (
    customer_analysis["total_spend"].sum()
)

avg_spend_per_customer = (
    total_spend / total_customers
)

print(f"顧客数: {total_customers:,}")
print(f"リピート顧客数: {repeat_customers:,}")
print(f"リピート率: {repeat_rate:.1%}")
print(f"総購入金額: ${total_spend:,.2f}")
print(f"顧客あたり平均購入金額: ${avg_spend_per_customer:,.2f}")

NameError: name 'customer_analysis' is not defined

In [10]:
customer_analysis["spend_segment"] = pd.qcut(
    customer_analysis["total_spend"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

customer_analysis["frequency_segment"] = pd.qcut(
    customer_analysis["order_count"].rank(method="first"),
    q=3,
    labels=["Low", "Medium", "High"]
)

display(
    customer_analysis[
        [
            "customer_id",
            "total_spend",
            "order_count",
            "spend_segment",
            "frequency_segment"
        ]
    ].head(10)
)

NameError: name 'customer_analysis' is not defined

In [11]:
segment_summary = (
    customer_analysis
    .groupby(["spend_segment", "frequency_segment"], observed=True)
    .agg(
        customers=("customer_id", "count"),
        total_spend=("total_spend", "sum"),
        avg_spend=("total_spend", "mean"),
        avg_orders=("order_count", "mean")
    )
    .reset_index()
)

segment_summary["sales_share"] = (
    segment_summary["total_spend"] / total_spend
)

display(
    segment_summary.style.format({
        "total_spend": "${:,.2f}",
        "avg_spend": "${:,.2f}",
        "avg_orders": "{:.2f}",
        "sales_share": "{:.1%}"
    })
)

NameError: name 'customer_analysis' is not defined

In [12]:
customer_type_summary = (
    customer_analysis
    .groupby("customer_type")
    .agg(
        customers=("customer_id", "count"),
        total_spend=("total_spend", "sum"),
        avg_spend=("total_spend", "mean"),
        avg_orders=("order_count", "mean")
    )
    .reset_index()
)

customer_type_summary["customer_share"] = (
    customer_type_summary["customers"] / total_customers
)

customer_type_summary["sales_share"] = (
    customer_type_summary["total_spend"] / total_spend
)

display(
    customer_type_summary.style.format({
        "total_spend": "${:,.2f}",
        "avg_spend": "${:,.2f}",
        "avg_orders": "{:.2f}",
        "customer_share": "{:.1%}",
        "sales_share": "{:.1%}"
    })
)

NameError: name 'customer_analysis' is not defined

In [13]:
country_summary = (
    customer_analysis
    .groupby("country")
    .agg(
        customers=("customer_id", "count"),
        total_spend=("total_spend", "sum"),
        avg_spend=("total_spend", "mean"),
        avg_orders=("order_count", "mean"),
        repeat_customers=("is_repeat", "sum")
    )
    .reset_index()
)

country_summary["repeat_rate"] = (
    country_summary["repeat_customers"]
    / country_summary["customers"]
)

country_summary["sales_share"] = (
    country_summary["total_spend"] / total_spend
)

display(
    country_summary.style.format({
        "total_spend": "${:,.2f}",
        "avg_spend": "${:,.2f}",
        "avg_orders": "{:.2f}",
        "repeat_rate": "{:.1%}",
        "sales_share": "{:.1%}"
    })
)

NameError: name 'customer_analysis' is not defined

In [14]:
plt.figure(figsize=(8, 5))

plt.bar(
    customer_type_summary["customer_type"],
    customer_type_summary["total_spend"]
)

plt.title("Total Spend by Customer Type")
plt.xlabel("Customer Type")
plt.ylabel("Total Spend (USD)")

for i, value in enumerate(
    customer_type_summary["total_spend"]
):
    plt.text(
        i,
        value,
        f"${value:,.0f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

NameError: name 'customer_type_summary' is not defined

<Figure size 800x500 with 0 Axes>

In [15]:
plt.figure(figsize=(8, 5))

plt.bar(
    country_summary["country"],
    country_summary["repeat_rate"]
)

plt.title("Repeat Customer Rate by Country")
plt.xlabel("Country")
plt.ylabel("Repeat Rate")

plt.ylim(
    0,
    max(country_summary["repeat_rate"]) * 1.3
)

for i, value in enumerate(
    country_summary["repeat_rate"]
):
    plt.text(
        i,
        value,
        f"{value:.1%}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

NameError: name 'country_summary' is not defined

<Figure size 800x500 with 0 Axes>

In [16]:
reference_date = pd.to_datetime(orders["order_date"]).max()

print("分析基準日:", reference_date)

NameError: name 'orders' is not defined

In [17]:
orders["order_date"] = pd.to_datetime(orders["order_date"])

rfm = (
    orders.groupby("customer_id")
    .agg(
        recency=("order_date", lambda x: (reference_date - x.max()).days),
        frequency=("order_id", "nunique"),
        monetary=("unit_price", lambda x: (
            x * orders.loc[x.index, "quantity"]
        ).sum())
    )
    .reset_index()
)

display(rfm.head(10))

NameError: name 'orders' is not defined

In [18]:
rfm["R_score"] = pd.qcut(
    rfm["recency"].rank(method="first", ascending=False),
    q=3,
    labels=[3, 2, 1]
).astype(int)

rfm["F_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    q=3,
    labels=[1, 2, 3]
).astype(int)

rfm["M_score"] = pd.qcut(
    rfm["monetary"].rank(method="first"),
    q=3,
    labels=[1, 2, 3]
).astype(int)

rfm["RFM_score"] = (
    rfm["R_score"].astype(str)
    + rfm["F_score"].astype(str)
    + rfm["M_score"].astype(str)
)

display(rfm.head(10))

NameError: name 'rfm' is not defined

In [19]:
def classify_customer(row):
    if row["R_score"] == 3 and row["F_score"] >= 2 and row["M_score"] >= 2:
        return "Loyal / High Value"
    elif row["R_score"] == 3:
        return "Recent"
    elif row["F_score"] == 3:
        return "Frequent"
    elif row["M_score"] == 3:
        return "High Spender"
    else:
        return "Low Engagement"

rfm["rfm_segment"] = rfm.apply(
    classify_customer,
    axis=1
)

display(
    rfm[
        [
            "customer_id",
            "recency",
            "frequency",
            "monetary",
            "RFM_score",
            "rfm_segment"
        ]
    ].head(15)
)

NameError: name 'rfm' is not defined

In [20]:
rfm_summary = (
    rfm.groupby("rfm_segment")
    .agg(
        customers=("customer_id", "count"),
        total_spend=("monetary", "sum"),
        avg_spend=("monetary", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_recency=("recency", "mean")
    )
    .reset_index()
)

rfm_summary["customer_share"] = (
    rfm_summary["customers"] / len(rfm)
)

rfm_summary["sales_share"] = (
    rfm_summary["total_spend"] / rfm["monetary"].sum()
)

display(
    rfm_summary.style.format({
        "total_spend": "${:,.2f}",
        "avg_spend": "${:,.2f}",
        "avg_frequency": "{:.2f}",
        "avg_recency": "{:.1f}",
        "customer_share": "{:.1%}",
        "sales_share": "{:.1%}"
    })
)

NameError: name 'rfm' is not defined

In [21]:
plt.figure(figsize=(9, 5))

plt.bar(
    rfm_summary["rfm_segment"],
    rfm_summary["customers"]
)

plt.title("Customer Count by RFM Segment")
plt.xlabel("RFM Segment")
plt.ylabel("Number of Customers")

plt.xticks(rotation=25, ha="right")

for i, value in enumerate(rfm_summary["customers"]):
    plt.text(
        i,
        value,
        f"{value:,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

NameError: name 'rfm_summary' is not defined

<Figure size 900x500 with 0 Axes>

In [22]:
plt.figure(figsize=(9, 5))

plt.bar(
    rfm_summary["rfm_segment"],
    rfm_summary["sales_share"]
)

plt.title("Sales Share by RFM Segment")
plt.xlabel("RFM Segment")
plt.ylabel("Sales Share")

plt.xticks(rotation=25, ha="right")

for i, value in enumerate(rfm_summary["sales_share"]):
    plt.text(
        i,
        value,
        f"{value:.1%}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

NameError: name 'rfm_summary' is not defined

<Figure size 900x500 with 0 Axes>

In [23]:
rfm_comparison = rfm_summary[
    [
        "rfm_segment",
        "customers",
        "customer_share",
        "total_spend",
        "sales_share"
    ]
].copy()

display(
    rfm_comparison.style.format({
        "customer_share": "{:.1%}",
        "total_spend": "${:,.2f}",
        "sales_share": "{:.1%}"
    })
)

NameError: name 'rfm_summary' is not defined

In [24]:
high_value_customers = rfm[
    rfm["rfm_segment"] == "Loyal / High Value"
].sort_values(
    ["monetary", "frequency"],
    ascending=[False, False]
)

display(
    high_value_customers[
        [
            "customer_id",
            "recency",
            "frequency",
            "monetary",
            "RFM_score",
            "rfm_segment"
        ]
    ]
)

NameError: name 'rfm' is not defined

In [25]:
top_segment = (
    rfm_summary
    .sort_values("sales_share", ascending=False)
    .iloc[0]
)

print("=== RFM Analysis Summary ===")
print(f"最大売上貢献セグメント: {top_segment['rfm_segment']}")
print(f"顧客数: {int(top_segment['customers']):,}")
print(f"顧客構成比: {top_segment['customer_share']:.1%}")
print(f"売上構成比: {top_segment['sales_share']:.1%}")
print(f"平均購入金額: ${top_segment['avg_spend']:,.2f}")
print(f"平均購入回数: {top_segment['avg_frequency']:.2f}")

NameError: name 'rfm_summary' is not defined

In [26]:
orders["sales_amount"] = (
    orders["quantity"] * orders["unit_price"]
)

product_summary = (
    orders
    .groupby("product_id")
    .agg(
        orders=("order_id", "nunique"),
        units_sold=("quantity", "sum"),
        total_sales=("sales_amount", "sum"),
        avg_unit_price=("unit_price", "mean")
    )
    .reset_index()
    .sort_values("total_sales", ascending=False)
)

product_summary["sales_share"] = (
    product_summary["total_sales"]
    / product_summary["total_sales"].sum()
)

display(
    product_summary.style.format({
        "total_sales": "${:,.2f}",
        "avg_unit_price": "${:,.2f}",
        "sales_share": "{:.1%}"
    })
)

NameError: name 'orders' is not defined

In [27]:
top_products = product_summary.head(10).copy()

display(
    top_products.style.format({
        "total_sales": "${:,.2f}",
        "avg_unit_price": "${:,.2f}",
        "sales_share": "{:.1%}"
    })
)

NameError: name 'product_summary' is not defined

In [28]:
plt.figure(figsize=(10, 6))

plt.barh(
    top_products["product_id"].astype(str)[::-1],
    top_products["total_sales"][::-1]
)

plt.title("Top 10 Products by Sales")
plt.xlabel("Total Sales (USD)")
plt.ylabel("Product ID")

plt.tight_layout()
plt.show()

NameError: name 'top_products' is not defined

<Figure size 1000x600 with 0 Axes>

In [29]:
top_units = (
    product_summary
    .sort_values("units_sold", ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_units["product_id"].astype(str)[::-1],
    top_units["units_sold"][::-1]
)

plt.title("Top 10 Products by Units Sold")
plt.xlabel("Units Sold")
plt.ylabel("Product ID")

plt.tight_layout()
plt.show()

NameError: name 'product_summary' is not defined

In [30]:
product_summary["revenue_per_order"] = (
    product_summary["total_sales"]
    / product_summary["orders"]
)

product_summary["revenue_per_unit"] = (
    product_summary["total_sales"]
    / product_summary["units_sold"]
)

display(
    product_summary[
        [
            "product_id",
            "orders",
            "units_sold",
            "total_sales",
            "sales_share",
            "revenue_per_order",
            "revenue_per_unit"
        ]
    ].sort_values(
        "total_sales",
        ascending=False
    ).style.format({
        "total_sales": "${:,.2f}",
        "sales_share": "{:.1%}",
        "revenue_per_order": "${:,.2f}",
        "revenue_per_unit": "${:,.2f}"
    })
)

NameError: name 'product_summary' is not defined

In [42]:
channel_summary = (
    orders
    .groupby("sales_channel")
    .agg(
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
        units_sold=("quantity", "sum"),
        total_sales=("sales_amount", "sum")
    )
    .reset_index()
)

channel_summary["sales_share"] = (
    channel_summary["total_sales"]
    / channel_summary["total_sales"].sum()
)

channel_summary["avg_order_value"] = (
    channel_summary["total_sales"]
    / channel_summary["orders"]
)

display(
    channel_summary.style.format({
        "total_sales": "${:,.2f}",
        "sales_share": "{:.1%}",
        "avg_order_value": "${:,.2f}"
    })
)

NameError: name 'orders' is not defined

In [43]:
plt.figure(figsize=(8, 5))

plt.bar(
    channel_summary["sales_channel"],
    channel_summary["total_sales"]
)

plt.title("Sales by Sales Channel")
plt.xlabel("Sales Channel")
plt.ylabel("Total Sales (USD)")

for i, value in enumerate(channel_summary["total_sales"]):
    plt.text(
        i,
        value,
        f"${value:,.0f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

NameError: name 'channel_summary' is not defined

<Figure size 800x500 with 0 Axes>

In [44]:
plt.figure(figsize=(8, 5))

plt.bar(
    channel_summary["sales_channel"],
    channel_summary["orders"]
)

plt.title("Orders by Sales Channel")
plt.xlabel("Sales Channel")
plt.ylabel("Number of Orders")

for i, value in enumerate(channel_summary["orders"]):
    plt.text(
        i,
        value,
        f"{int(value):,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

NameError: name 'channel_summary' is not defined

<Figure size 800x500 with 0 Axes>

In [45]:
plt.figure(figsize=(8, 5))

plt.bar(
    channel_summary["sales_channel"],
    channel_summary["avg_order_value"]
)

plt.title("Average Order Value by Sales Channel")
plt.xlabel("Sales Channel")
plt.ylabel("Average Order Value (USD)")

for i, value in enumerate(channel_summary["avg_order_value"]):
    plt.text(
        i,
        value,
        f"${value:,.2f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

NameError: name 'channel_summary' is not defined

<Figure size 800x500 with 0 Axes>

In [46]:
best_sales_channel = (
    channel_summary
    .sort_values("total_sales", ascending=False)
    .iloc[0]
)

best_aov_channel = (
    channel_summary
    .sort_values("avg_order_value", ascending=False)
    .iloc[0]
)

print("=== Sales Channel Analysis ===")
print(
    f"売上最大チャネル: "
    f"{best_sales_channel['sales_channel']}"
)
print(
    f"売上: "
    f"${best_sales_channel['total_sales']:,.2f}"
)
print(
    f"売上構成比: "
    f"{best_sales_channel['sales_share']:.1%}"
)
print(
    f"平均注文金額が最大のチャネル: "
    f"{best_aov_channel['sales_channel']}"
)
print(
    f"平均注文金額: "
    f"${best_aov_channel['avg_order_value']:,.2f}"
)

NameError: name 'channel_summary' is not defined

NameError: name 'orders' is not defined

NameError: name 'channel_summary' is not defined

<Figure size 800x500 with 0 Axes>